# Additional features in clustering only: GMM and DBSCAN

Keep the final classifier on the original 150 embeddings, with its frozen 157-row scaler, C=1000, normalized fitting-weight total 157 per reason, cutoff 0.5, and highest-probability fallback. Only the label-generation clustering input changes. Known labels and raw weight 1 stay fixed; the acceptance and weight-budget rules are unchanged.

Use the original 150-dimensional standardized clustering embeddings. Add one group at a time: existing **71 disease indicators** (minimum training frequency 5), or existing **2,000-term criteria TF-IDF** compressed by a new training-only **20-component TruncatedSVD**. Existing vocabularies/IDF are reused without refitting. Center each added block and normalize its aggregate training RMS to match the embedding block, then apply relative input-scale multipliers **0.10,0.25,1.0**. Their added aggregate variance is 1%,6.25%,100% of the embedding variance. These control input geometry; a GMM's covariance fitting and regularization mean they are not literal likelihood feature-importance weights.

Seven representations: embeddings alone, plus two groups × three scales. **GMM:** keep K=10, tied covariance, and the original parameters (7 fits). **DBSCAN:** keep min_samples=5 and recompute epsilon at the 25th/50th/75th percentiles of each representation's fifth-neighbor training distances, including self as in the original implementation (21 fits). Total **28 configurations**. New distances require new radii; compare both against the best embedding-only family control and the same radius-quantile control.

All new preprocessing/clustering use the 1,171 training trials only. The classifier still uses only its accepted original/generated training labels. No extra classifier features, validation/test fitting, self-training, final validation-label refit, Kaggle submission, commit, or tag.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import warnings
import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from threadpoolctl import threadpool_limits

SEED = 42
REASONS = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
FEATURES = [f"emb_{i}" for i in range(150)]
LABEL_RULE = {'minimum_support': 3.0, 'prior_strength': 2.0, 'positive_threshold': 0.6, 'negative_threshold': 0.1, 'maximum_generated_weight': 0.25, 'maximum_class_fraction': 0.5}
CONTEXT_SCALES = [0.10,0.25,1.0]
DBSCAN_QUANTILES = [0.25,0.50,0.75]
MIN_SAMPLES = 5
TEXT_SVD_COMPONENTS = 20
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import DBSCAN
from scipy import sparse
import ast
from sklearn.base import clone
from sklearn.decomposition import PCA
cwd = Path.cwd()
DATA = next(q for q in [cwd, cwd/"part1", cwd/"AML-Midterm"/"part1"]
            if (q/"part1_train.csv").is_file() or (q/"data"/"part1_train.csv").is_file()).resolve()
INPUT_DIR = DATA if (DATA/"part1_train.csv").is_file() else DATA/"data"
OUT = DATA/"experiments"/"clustering_feature_screen"
for folder in ["datasets", "label_audits", "clusters", "clustering_models", "classifier_models", "validation_predictions", "submissions", "labeling_models", "clustering_inputs", "representations", "profiles", "clustering_outputs"]:
    (OUT/folder).mkdir(parents=True, exist_ok=True)
train = pd.read_csv(INPUT_DIR/"part1_train.csv")
valid = pd.read_csv(INPUT_DIR/"part1_val.csv")
X = train[FEATURES].to_numpy()
X_valid = valid[FEATURES].to_numpy()
truth = valid[REASONS].to_numpy(dtype=int)
original_y = train[REASONS].to_numpy(dtype=int)
known = train[REASONS].isin([0,1]).all(axis=1).to_numpy()
assert known.sum()==157 and len(train)==1171 and len(valid)==78
assert train.ID.is_unique and valid.ID.is_unique and set(train.ID).isdisjoint(valid.ID)
assert np.isfinite(X).all() and np.isfinite(X_valid).all()
assert np.isin(truth,[0,1]).all()
baseline = joblib.load(DATA/"experiments/kmeans/models/baseline_C1000.joblib")
legacy_packet = joblib.load(DATA/"experiments/kmeans/models/k02.joblib")
classifier_scaler = baseline.named_steps["scaler"]
cluster_scaler = legacy_packet["kmeans_scaler"]
assert classifier_scaler.n_samples_seen_==157 and cluster_scaler.n_samples_seen_==1171
XS = classifier_scaler.transform(X)
VS = classifier_scaler.transform(X_valid)
XC = cluster_scaler.transform(X)
protected_paths = [INPUT_DIR/"part1_train.csv", INPUT_DIR/"part1_val.csv"] + sorted(DATA.glob("*submission.csv")) + [
    DATA/"experiments/kmeans/models/k02.joblib", DATA/"experiments/kmeans/datasets/k02_train.csv"
]
protected_hashes = {str(q.relative_to(DATA)):hashlib.sha256(q.read_bytes()).hexdigest() for q in protected_paths}
print("Using 150 features; 157 known-label and 1,014 unlabeled training trials; 78 validation trials.")

Using 150 features; 157 known-label and 1,014 unlabeled training trials; 78 validation trials.


In [2]:
def label_predictions(probabilities):
    probabilities = np.asarray(probabilities)
    assert probabilities.shape[1]==5 and np.isfinite(probabilities).all()
    assert ((probabilities>=0)&(probabilities<=1)).all()
    prediction=(probabilities>=0.5).astype(int)
    empty=prediction.sum(axis=1)==0
    ids=np.flatnonzero(empty)
    prediction[ids, probabilities[ids].argmax(axis=1)]=1
    return prediction,empty


def score_probabilities(probabilities):
    predicted,empty=label_predictions(probabilities)
    return {"sample_f1":f1_score(truth,predicted,average="samples",zero_division=0),
            "macro_f1":f1_score(truth,predicted,average="macro",zero_division=0),
            "fallback_trials":int(empty.sum()),
            **dict(zip([f"f1_{r}" for r in REASONS],f1_score(truth,predicted,average=None,zero_division=0)))},predicted,empty


def save_predictions(name, probabilities):
    metrics,predicted,empty=score_probabilities(probabilities)
    output=valid[["ID"]+REASONS].rename(columns={r:f"actual_{r}" for r in REASONS}).copy()
    for j,r in enumerate(REASONS):
        output[f"probability_{r}"]=probabilities[:,j]
        output[f"predicted_{r}"]=predicted[:,j]
    output["used_fallback"]=empty
    output.to_csv(OUT/"validation_predictions"/f"{name}.csv",index=False)
    return metrics


DATASETS={}
LABEL_COUNTS=[]
FIT_CACHE={}


def save_dataset(name, targets, weights, probabilities=None, component=None, support=None):
    assert targets.shape==weights.shape==(len(train),5)
    np.testing.assert_array_equal(targets[known],original_y[known])
    np.testing.assert_array_equal(weights[known],np.ones((known.sum(),5)))
    assert np.isin(targets,[-100,0,1]).all() and (weights>=0).all()
    assert np.array_equal(weights>0,targets!=-100)
    frame=train[["ID"]+FEATURES].copy()
    audit=train[["ID"]].copy()
    for j,r in enumerate(REASONS):
        source=np.where(known,"original",np.where(weights[:,j]>0,"generated","unknown"))
        frame[r]=targets[:,j]
        frame[f"weight_{r}"]=weights[:,j]
        audit[r]=targets[:,j]
        audit[f"original_{r}"]=original_y[:,j]
        audit[f"weight_{r}"]=weights[:,j]
        audit[f"label_source_{r}"]=source
        if probabilities is not None:audit[f"estimated_probability_{r}"]=probabilities[:,j]
        pos=(~known)&(targets[:,j]==1)
        neg=(~known)&(targets[:,j]==0)
        LABEL_COUNTS.append({"dataset":name,"reason":r,
            "known_positives":int(original_y[known,j].sum()),
            "known_negatives":int((original_y[known,j]==0).sum()),
            "generated_positives":int(pos.sum()),"generated_negatives":int(neg.sum()),
            "generated_positive_weight":float(weights[pos,j].sum()),
            "generated_negative_weight":float(weights[neg,j].sum())})
    if component is not None:audit["component"]=component
    if support is not None:audit["effective_labeled_support"]=support
    frame["included_for_any_reason"]=(weights>0).any(axis=1)
    frame.to_csv(OUT/"datasets"/f"{name}.csv",index=False)
    audit.to_csv(OUT/"label_audits"/f"{name}.csv",index=False)
    DATASETS[name]={"targets":targets.copy(),"weights":weights.copy(),
                    "generated_trials":int(((weights>0).any(axis=1)&~known).sum())}
    return DATASETS[name]


def partial_labels(responsibilities):
    assert responsibilities.shape[0]==len(train)
    np.testing.assert_allclose(responsibilities.sum(axis=1),1,atol=1e-9)
    component_support=responsibilities[known].sum(axis=0)
    positives=responsibilities[known].T@original_y[known]
    prior=original_y[known].mean(axis=0)
    rate=(positives+LABEL_RULE["prior_strength"]*prior)/(component_support[:,None]+LABEL_RULE["prior_strength"])
    probabilities=responsibilities@rate
    support=responsibilities@component_support
    targets=np.full(original_y.shape,-100,dtype=int)
    weights=np.zeros(original_y.shape,dtype=float)
    targets[known]=original_y[known]
    weights[known]=1.0
    eligible=(~known)&(support>=LABEL_RULE["minimum_support"])
    for j in range(5):
        positive=eligible&(probabilities[:,j]>=LABEL_RULE["positive_threshold"])
        negative=eligible&(probabilities[:,j]<=LABEL_RULE["negative_threshold"])
        npos,nneg=int(positive.sum()),int(negative.sum())
        kp=int(original_y[known,j].sum());kn=int(known.sum()-kp)
        if not npos or not nneg:continue
        fraction=min(LABEL_RULE["maximum_class_fraction"],
                     LABEL_RULE["maximum_generated_weight"]*npos/kp,
                     LABEL_RULE["maximum_generated_weight"]*nneg/kn)
        targets[positive,j]=1;targets[negative,j]=0
        weights[positive,j]=fraction*kp/npos
        weights[negative,j]=fraction*kn/nneg
        assert weights[~known,j].max()<=LABEL_RULE["maximum_generated_weight"]+1e-12
        np.testing.assert_allclose(weights[positive,j].sum()/kp,weights[negative,j].sum()/kn,atol=1e-12)
    components=pd.DataFrame({"component":np.arange(len(component_support)),"effective_labeled_support":component_support,
                             "total_membership_mass":responsibilities.sum(axis=0)})
    for j,r in enumerate(REASONS):
        components[f"weighted_known_positive_{r}"]=positives[:,j]
        components[f"estimated_probability_{r}"]=rate[:,j]
    return targets,weights,probabilities,support,components


def fit_reason_models(dataset_name, kind="logistic", parameter=1000.0):
    dataset=DATASETS[dataset_name]
    estimators=[]
    for j in range(5):
        y=dataset["targets"][:,j];w=dataset["weights"][:,j]
        included=w>0
        assert np.isin(y[included],[0,1]).all() and included[known].all()
        key=(kind,float(parameter),j,hashlib.sha256(y.tobytes()+w.tobytes()).hexdigest())
        if key not in FIT_CACHE:
            if kind=="logistic":
                model=LogisticRegression(C=float(parameter),solver="lbfgs",max_iter=5000,tol=1e-4,random_state=SEED)
                normalized=w[included]*known.sum()/w[included].sum()
                with warnings.catch_warnings():
                    warnings.simplefilter("error",ConvergenceWarning)
                    model.fit(XS[included],y[included],sample_weight=normalized)
            else:
                model={"neighbors":NearestNeighbors(n_neighbors=min(int(parameter),int(included.sum())),algorithm="brute").fit(XS[included]),
                       "target":y[included].copy(),"sample_weight":w[included].copy()}
            FIT_CACHE[key]=model
        estimators.append(FIT_CACHE[key])
    return {"kind":kind,"parameter":float(parameter),"scaler":classifier_scaler,"estimators":estimators,
            "features":FEATURES,"reasons":REASONS,"dataset":dataset_name,
            "logistic_weight_normalization_total":int(known.sum())}


def predict_reason_models(model, raw_features):
    scaled=model["scaler"].transform(np.asarray(raw_features))
    output=[]
    for estimator in model["estimators"]:
        if model["kind"] in ["logistic", "rbf_svm", "gaussian_nb"]:
            positive=list(estimator.classes_).index(1)
            probability=estimator.predict_proba(scaled)[:,positive]
        else:
            distance,indices=estimator["neighbors"].kneighbors(scaled)
            zero=distance==0
            exact=zero.any(axis=1)
            distance_weight=np.divide(1.,distance,out=np.zeros_like(distance),where=distance!=0)
            distance_weight[exact]=zero[exact].astype(float)
            vote=distance_weight*estimator["sample_weight"][indices]
            probability=(vote*estimator["target"][indices]).sum(axis=1)/vote.sum(axis=1)
        output.append(probability)
    probabilities=np.column_stack(output)
    assert np.isfinite(probabilities).all()
    return np.clip(probabilities,0,1)


known_y=np.full_like(original_y,-100);known_y[known]=original_y[known]
known_w=np.zeros(original_y.shape);known_w[known]=1.0
save_dataset("known_only",known_y,known_w)
with threadpool_limits(limits=1):
    reproduced=fit_reason_models("known_only")
np.testing.assert_allclose(predict_reason_models(reproduced,X_valid),baseline.predict_proba(X_valid),rtol=1e-8,atol=1e-10)
reference_rows=[]
for name,model,kaggle in [("submitted_supervised",baseline,0.87072),("submitted_k02",legacy_packet["classifier"],0.87393)]:
    metrics=save_predictions(name,model.predict_proba(X_valid))
    reference_rows.append({"experiment":name,"dataset":"frozen reference","classifier":"logistic","parameter":1000.0,
                           "generated_trials":0 if name=="submitted_supervised" else 388,**metrics,"kaggle_score":kaggle})
print("Confirmed that the new per-reason trainer reproduces the supervised reference.")

Confirmed that the new per-reason trainer reproduces the supervised reference.


In [3]:
SOURCE=DATA/"experiments/method_comparison"
gmm_packet=joblib.load(SOURCE/"clustering_models/gmm_tied_k10.joblib")
gmm_reference=joblib.load(SOURCE/"classifier_models/gmm_tied_k10__logistic_C1000.joblib")
gmm_reference_probability=predict_reason_models(gmm_reference,X_valid)
gmm_reference_metrics=save_predictions("gmm_original_reference",gmm_reference_probability)
np.testing.assert_allclose(gmm_reference_metrics["sample_f1"],0.905982905982906,atol=1e-14)
gmm_reference_frame=pd.read_csv(SOURCE/"datasets/gmm_tied_k10.csv",float_precision="round_trip")
assert gmm_reference_frame.ID.tolist()==train.ID.tolist()
assert gmm_packet["clustering"].n_components==10 and gmm_packet["clustering"].covariance_type=="tied"
assert gmm_packet["label_rule"]==LABEL_RULE
np.testing.assert_array_equal(gmm_packet["scaler"].mean_,cluster_scaler.mean_)
np.testing.assert_array_equal(gmm_packet["scaler"].scale_,cluster_scaler.scale_)
np.testing.assert_array_equal(gmm_reference["scaler"].mean_,classifier_scaler.mean_)
DB_SOURCE=DATA/"experiments/additional_methods"
db_record=json.loads((DB_SOURCE/"submission_manifest.json").read_text())
db_model=joblib.load(DB_SOURCE/db_record["model_file"])
db_metrics=save_predictions("dbscan_current_reference",predict_reason_models(db_model,X_valid))
np.testing.assert_allclose(db_metrics["sample_f1"],0.9102564102564102,atol=1e-14)
for p in [SOURCE/"clustering_models/gmm_tied_k10.joblib",SOURCE/"classifier_models/gmm_tied_k10__logistic_C1000.joblib",
          SOURCE/"datasets/gmm_tied_k10.csv",DB_SOURCE/db_record["model_file"],DB_SOURCE/db_record["submission_file"]]:
    protected_hashes[str(p.relative_to(DATA))]=hashlib.sha256(p.read_bytes()).hexdigest()
db_reference_frame=pd.read_csv(DB_SOURCE/"datasets/dbscan_m5_q25.csv",float_precision="round_trip")
assert db_reference_frame.ID.tolist()==train.ID.tolist()
db_label_packet=joblib.load(DB_SOURCE/"labeling_models/dbscan_m5_q25.joblib")
assert db_label_packet["model"].min_samples==5
np.testing.assert_array_equal(db_label_packet["scaler"].mean_,cluster_scaler.mean_)
FEATURE_SOURCE=DATA/"experiments/classifier_features"
CONTEXT_MODELS={name:joblib.load(FEATURE_SOURCE/"models"/f"{name}.joblib") for name in ["diseases_min5","criteria_min3"]}
for path in [DB_SOURCE/"datasets/dbscan_m5_q25.csv",DB_SOURCE/"labeling_models/dbscan_m5_q25.joblib"]+[
    FEATURE_SOURCE/"models"/f"{name}.joblib" for name in CONTEXT_MODELS]:
    protected_hashes[str(path.relative_to(DATA))]=hashlib.sha256(path.read_bytes()).hexdigest()
ROWS=[];MODELS={};REPRESENTATIONS={};RADIUS_ROWS=[];DISTANCE_ROWS=[];MATCHED_BASELINES={}
FAMILY_BASELINES={"gmm":gmm_reference_metrics["sample_f1"],"dbscan":db_metrics["sample_f1"]}


In [4]:
def disease_lists(frame):
    def parse(value):
        if pd.isna(value) or not str(value).strip():return []
        values=ast.literal_eval(value)
        assert isinstance(values,list)
        return sorted({str(v).strip().lower() for v in values if str(v).strip()})
    return frame.diseases_cleaned.map(parse).tolist()


def pathway_lists(frame):
    return frame.drug_pathways.fillna("").map(lambda s:sorted({v.strip() for v in str(s).split(";") if v.strip()})).tolist()


def missing_pathways(frame):
    return frame.drug_pathways.fillna("").astype(str).str.strip().eq("").to_numpy(dtype=float)[:,None]

def transform_block(block,frame):
    kind=block["kind"]
    if kind=="none":return sparse.csr_matrix((len(frame),0))
    if kind=="missing_only":return sparse.csr_matrix(missing_pathways(frame))
    if kind=="criteria":return block["vectorizer"].transform(frame.criteria.fillna("")).tocsr()
    values=disease_lists(frame) if kind=="diseases" else pathway_lists(frame)
    allowed=set(block["encoder"].classes_)
    result=block["encoder"].transform([[v for v in row if v in allowed] for row in values]).astype(float).tocsr()
    if kind=="pathways":result=sparse.hstack([result,sparse.csr_matrix(missing_pathways(frame))],format="csr")
    assert result.shape[1]==len(block["feature_names"])
    assert np.isin(result.data,[0,1]).all()
    return result


def targets_from_probabilities(probabilities,support,valid_probability=None):
    valid_probability=np.ones_like(probabilities,dtype=bool) if valid_probability is None else valid_probability
    targets=np.full_like(original_y,-100)
    weights=np.zeros(original_y.shape)
    targets[known]=original_y[known];weights[known]=1.0
    eligible=(~known)[:,None]&(np.asarray(support)>=LABEL_RULE["minimum_support"])[:,None]&valid_probability
    for j in range(5):
        positive=eligible[:,j]&(probabilities[:,j]>=LABEL_RULE["positive_threshold"])
        negative=eligible[:,j]&(probabilities[:,j]<=LABEL_RULE["negative_threshold"])
        np_,nn=int(positive.sum()),int(negative.sum())
        kp=int(original_y[known,j].sum());kn=int(known.sum()-kp)
        if not np_ or not nn:continue
        fraction=min(LABEL_RULE["maximum_class_fraction"],LABEL_RULE["maximum_generated_weight"]*np_/kp,
                     LABEL_RULE["maximum_generated_weight"]*nn/kn)
        targets[positive,j]=1;targets[negative,j]=0
        weights[positive,j]=fraction*kp/np_;weights[negative,j]=fraction*kn/nn
        assert weights[~known,j].max()<=0.25+1e-12
        np.testing.assert_allclose(weights[positive,j].sum()/kp,weights[negative,j].sum()/kn,atol=1e-12)
    return targets,weights


def transform_context(context,frame):
    block=transform_block(context["preprocessor"],frame)
    raw=context["svd"].transform(block) if context["svd"] is not None else block.toarray()
    result=(raw-context["training_mean"])*context["rms_normalizer"]*context["relative_scale"]
    assert np.isfinite(result).all()
    return result


def transform_clustering(packet,frame):
    embeddings=packet["scaler"].transform(frame[FEATURES].to_numpy())
    return embeddings if packet["context"] is None else np.column_stack([embeddings,transform_context(packet["context"],frame)])

embedding_rms=float(np.sqrt(np.mean(np.sum(XC**2,axis=1))))
REPRESENTATIONS["embeddings_only"]={"matrix":XC.copy(),"context":None,"feature_group":"none","relative_scale":0.0}
with threadpool_limits(limits=1):
    for group,source_name in [("diseases","diseases_min5"),("criteria","criteria_min3")]:
        preprocessor=CONTEXT_MODELS[source_name]["block"]
        raw_block=transform_block(preprocessor,train)
        if group=="criteria":
            svd=TruncatedSVD(n_components=TEXT_SVD_COMPONENTS,n_iter=7,random_state=SEED).fit(raw_block)
            raw=svd.transform(raw_block)
            svd_retained=float(svd.explained_variance_ratio_.sum())
            print("Criteria SVD explained variance:",svd_retained)
        else:svd=None;raw=raw_block.toarray();svd_retained=None
        mean=raw.mean(axis=0);centered=raw-mean
        rms=float(np.sqrt(np.mean(np.sum(centered**2,axis=1))))
        assert rms>0
        normalizer=embedding_rms/rms
        pd.DataFrame([{"group":group,"raw_columns":raw_block.shape[1],"clustering_columns":raw.shape[1],
                       "raw_centered_rms":rms,"embedding_rms":embedding_rms,"rms_normalizer":normalizer,
                       "svd_explained_variance":svd_retained,"nonempty_raw_training_rows":int((raw_block.getnnz(axis=1)>0).sum())}]).to_csv(OUT/"profiles"/f"{group}_geometry.csv",index=False)
        for scale in CONTEXT_SCALES:
            name=f"{group}_scale{scale:g}".replace(".","p")
            context={"preprocessor":preprocessor,"svd":svd,"training_mean":mean,"rms_normalizer":normalizer,
                     "relative_scale":scale,"source_representation":source_name,"fit_rows":1171}
            added=transform_context(context,train)
            np.testing.assert_allclose(np.mean(np.sum(added**2,axis=1))/embedding_rms**2,scale**2,rtol=1e-10,atol=1e-12)
            REPESENTATION_MATRIX=np.column_stack([XC,added])
            REPRESENTATIONS[name]={"matrix":REPESENTATION_MATRIX,"context":context,"feature_group":group,"relative_scale":scale}
for name,representation in REPRESENTATIONS.items():
    matrix=representation["matrix"]
    np.testing.assert_array_equal(matrix[:,:150],XC)
    np.savez_compressed(OUT/"clustering_inputs"/f"{name}.npz",ID=train.ID.to_numpy(dtype=str),coordinates=matrix)
    joblib.dump({key:value for key,value in representation.items() if key!="matrix"},OUT/"representations"/f"{name}.joblib",compress=3)
    print(name,"clustering dimensions",matrix.shape[1],"classifier dimensions",150)


Criteria SVD explained variance: 0.19562064966653805


embeddings_only clustering dimensions 150 classifier dimensions 150
diseases_scale0p1 clustering dimensions 221 classifier dimensions 150
diseases_scale0p25 clustering dimensions 221 classifier dimensions 150
diseases_scale1 clustering dimensions 221 classifier dimensions 150
criteria_scale0p1 clustering dimensions 170 classifier dimensions 150


criteria_scale0p25 clustering dimensions 170 classifier dimensions 150
criteria_scale1 clustering dimensions 170 classifier dimensions 150


In [5]:
def record_model(family,representation_name,cluster_model,y,w,probability,support,components,assignments,extra):
    representation=REPRESENTATIONS[representation_name]
    suffix=f"_q{round(extra['eps_quantile']*100):02d}" if family=="dbscan" else ""
    name=f"{family}_{representation_name}{suffix}"
    dataset=save_dataset(name,y,w,probability,assignments,support)
    is_reference=representation_name=="embeddings_only" and (family=="gmm" or extra["eps_quantile"]==0.25)
    if is_reference:
        frame=gmm_reference_frame if family=="gmm" else db_reference_frame
        np.testing.assert_array_equal(y,frame[REASONS].to_numpy(dtype=int))
        np.testing.assert_allclose(w,frame[[f"weight_{r}" for r in REASONS]].to_numpy(),rtol=1e-12,atol=1e-14)
    components.to_csv(OUT/"clusters"/f"{name}.csv",index=False)
    packet={"family":family,"clustering":cluster_model,"scaler":cluster_scaler,"context":representation["context"],
            "features":FEATURES,"representation":representation_name,"label_rule":LABEL_RULE.copy(),"fit_rows":1171,**extra}
    joblib.dump(packet,OUT/"labeling_models"/f"{name}.joblib",compress=3)
    np.savez_compressed(OUT/"clustering_outputs"/f"{name}.npz",ID=train.ID.to_numpy(dtype=str),assignments=assignments)
    with threadpool_limits(limits=1):classifier=fit_reason_models(name,"logistic",1000)
    classifier["labeling_model_file"]=f"labeling_models/{name}.joblib";classifier["extra_classifier_features"]=False
    assert all(estimator.n_features_in_==150 for estimator in classifier["estimators"])
    probs=predict_reason_models(classifier,X_valid);metrics=save_predictions(name,probs)
    if is_reference:
        original=gmm_reference_probability if family=="gmm" else predict_reason_models(db_model,X_valid)
        np.testing.assert_allclose(probs,original,rtol=1e-8,atol=1e-10)
    joblib.dump(classifier,OUT/"classifier_models"/f"{name}.joblib",compress=3);MODELS[name]=classifier
    matched_key=(family,extra.get("eps_quantile"))
    if representation_name=="embeddings_only":MATCHED_BASELINES[matched_key]=metrics["sample_f1"]
    ROWS.append({"experiment":name,"family":family,"representation":representation_name,"feature_group":representation["feature_group"],
                 "relative_scale":representation["relative_scale"],"clustering_dimensions":representation["matrix"].shape[1],
                 "classifier_dimensions":150,"generated_trials":dataset["generated_trials"],
                 "generated_reason_slots":int(((w>0)&(~known)[:,None]).sum()),**extra,**metrics,
                 "delta_vs_family_embedding_best":metrics["sample_f1"]-FAMILY_BASELINES[family],
                 "delta_vs_matched_embedding_control":metrics["sample_f1"]-MATCHED_BASELINES[matched_key],
                 "delta_vs_current_leader":metrics["sample_f1"]-db_metrics["sample_f1"],
                 "is_current":family=="dbscan" and representation_name=="embeddings_only" and extra["eps_quantile"]==0.25,
                 "kaggle_score":np.nan})
    pd.DataFrame(ROWS).to_csv(OUT/"comparison.csv",index=False)
    pd.DataFrame(LABEL_COUNTS).to_csv(OUT/"label_counts.csv",index=False)
    print(f"{name}: sample F1={metrics['sample_f1']:.6f}, macro F1={metrics['macro_f1']:.6f}, "
          f"generated={dataset['generated_trials']}, change vs leader={ROWS[-1]['delta_vs_current_leader']:+.6f}",flush=True)


def run_gmm(representation_name):
    matrix=REPRESENTATIONS[representation_name]["matrix"]
    with threadpool_limits(limits=1):
        model=clone(gmm_packet["clustering"])
        with warnings.catch_warnings():
            warnings.simplefilter("error",ConvergenceWarning);model.fit(matrix)
        assert model.converged_
        responsibility=model.predict_proba(matrix)
        y,w,probability,support,components=partial_labels(responsibility)
    record_model("gmm",representation_name,model,y,w,probability,support,components,responsibility.argmax(axis=1),
                 {"clusters":10,"covariance_type":"tied","gmm_iterations":int(model.n_iter_)})


def dbscan_labels(model):
    ids=model.labels_;unique=np.unique(ids[ids>=0])
    if len(unique):
        membership=np.column_stack([ids==c for c in unique]).astype(float)
        counts=membership[known].sum(axis=0);positive=membership[known].T@original_y[known]
        rates=(positive+LABEL_RULE["prior_strength"]*original_y[known].mean(axis=0))/(counts[:,None]+LABEL_RULE["prior_strength"])
        probability=membership@rates;support=membership@counts
        components=pd.DataFrame({"cluster":unique,"training_trials":membership.sum(axis=0),"known_support":counts})
        for j,r in enumerate(REASONS):components[f"probability_{r}"]=rates[:,j]
    else:
        probability=np.tile(original_y[known].mean(axis=0),(len(train),1));support=np.zeros(len(train))
        components=pd.DataFrame(columns=["cluster","training_trials","known_support"])
    valid_probability=np.repeat((ids>=0)[:,None],5,axis=1)
    y,w=targets_from_probabilities(probability,support,valid_probability)
    assert (w[(ids<0)&~known]==0).all()
    return y,w,np.where(valid_probability,probability,np.nan),support,components,ids


def run_dbscan(representation_name):
    matrix=REPRESENTATIONS[representation_name]["matrix"]
    with threadpool_limits(limits=1):
        distances=NearestNeighbors(n_neighbors=MIN_SAMPLES,algorithm="brute").fit(matrix).kneighbors(matrix)[0][:,-1]
        pd.DataFrame({"ID":train.ID,"fifth_neighbor_distance_including_self":distances}).to_csv(OUT/"profiles"/f"{representation_name}_neighbor_distances.csv",index=False)
        for quantile in DBSCAN_QUANTILES:
            eps=max(float(np.quantile(distances,quantile)),1e-6)
            density=DBSCAN(eps=eps,min_samples=MIN_SAMPLES,metric="euclidean",algorithm="brute",n_jobs=1).fit(matrix)
            y,w,probability,support,components,ids=dbscan_labels(density)
            extra={"eps":eps,"eps_quantile":quantile,"min_samples":MIN_SAMPLES,
                   "clusters":int(np.unique(ids[ids>=0]).size),"noise_trials":int((ids<0).sum())}
            RADIUS_ROWS.append({"representation":representation_name,**extra})
            record_model("dbscan",representation_name,density,y,w,probability,support,components,ids,extra)
    pd.DataFrame(RADIUS_ROWS).to_csv(OUT/"dbscan_radius_audit.csv",index=False)

run_gmm("embeddings_only")
run_dbscan("embeddings_only")


gmm_embeddings_only: sample F1=0.905983, macro F1=0.811626, generated=696, change vs leader=-0.004274


dbscan_embeddings_only_q25: sample F1=0.910256, macro F1=0.811421, generated=133, change vs leader=+0.000000


dbscan_embeddings_only_q50: sample F1=0.888889, macro F1=0.789590, generated=483, change vs leader=-0.021368


dbscan_embeddings_only_q75: sample F1=0.893162, macro F1=0.804324, generated=0, change vs leader=-0.017094


In [6]:
for name in REPRESENTATIONS:
    if name!="embeddings_only":run_gmm(name)


gmm_diseases_scale0p1: sample F1=0.893162, macro F1=0.807626, generated=871, change vs leader=-0.017094


gmm_diseases_scale0p25: sample F1=0.846154, macro F1=0.741988, generated=854, change vs leader=-0.064103


gmm_diseases_scale1: sample F1=0.884615, macro F1=0.768865, generated=449, change vs leader=-0.025641


gmm_criteria_scale0p1: sample F1=0.888889, macro F1=0.802384, generated=876, change vs leader=-0.021368


gmm_criteria_scale0p25: sample F1=0.824786, macro F1=0.695511, generated=1013, change vs leader=-0.085470


gmm_criteria_scale1: sample F1=0.871795, macro F1=0.788407, generated=637, change vs leader=-0.038462


In [7]:
for name in REPRESENTATIONS:
    if name!="embeddings_only":run_dbscan(name)


dbscan_diseases_scale0p1_q25: sample F1=0.897436, macro F1=0.803884, generated=168, change vs leader=-0.012821


dbscan_diseases_scale0p1_q50: sample F1=0.888889, macro F1=0.789590, generated=479, change vs leader=-0.021368


dbscan_diseases_scale0p1_q75: sample F1=0.893162, macro F1=0.804324, generated=0, change vs leader=-0.017094


dbscan_diseases_scale0p25_q25: sample F1=0.884615, macro F1=0.793668, generated=169, change vs leader=-0.025641


dbscan_diseases_scale0p25_q50: sample F1=0.910256, macro F1=0.815391, generated=515, change vs leader=+0.000000


dbscan_diseases_scale0p25_q75: sample F1=0.893162, macro F1=0.804324, generated=0, change vs leader=-0.017094


... (8 outputs left out)

dbscan_criteria_scale0p25_q75: sample F1=0.893162, macro F1=0.804324, generated=0, change vs leader=-0.017094


dbscan_criteria_scale1_q25: sample F1=0.901709, macro F1=0.803884, generated=268, change vs leader=-0.008547


dbscan_criteria_scale1_q50: sample F1=0.893162, macro F1=0.804324, generated=0, change vs leader=-0.017094


dbscan_criteria_scale1_q75: sample F1=0.893162, macro F1=0.804324, generated=0, change vs leader=-0.017094


In [8]:
comparison=pd.DataFrame(ROWS);assert len(comparison)==28
assert comparison.groupby("family").size().to_dict()=={"dbscan":21,"gmm":7}
ranked=comparison.sort_values(["sample_f1","is_current","clustering_dimensions","relative_scale","experiment"],
                             ascending=[False,False,True,True,True],kind="stable")
best=ranked.iloc[0]
best_by_group=ranked.groupby(["family","feature_group"],sort=False).head(1)
best_by_group.to_csv(OUT/"best_by_family_group.csv",index=False)
best_by_representation=ranked.groupby(["family","representation"],sort=False).head(1)
best_by_representation.to_csv(OUT/"best_by_representation.csv",index=False)
comparison.query("representation=='embeddings_only'").to_csv(OUT/"embedding_controls.csv",index=False)
selected=joblib.load(OUT/"classifier_models"/f"{best.experiment}.joblib")
selected_probability=predict_reason_models(selected,X_valid)
for row in comparison.itertuples(index=False):
    classifier=joblib.load(OUT/"classifier_models"/f"{row.experiment}.joblib")
    frame=pd.read_csv(OUT/"validation_predictions"/f"{row.experiment}.csv",float_precision="round_trip")
    assert frame.ID.tolist()==valid.ID.tolist()
    probs=frame[[f"probability_{r}" for r in REASONS]].to_numpy();prediction=frame[[f"predicted_{r}" for r in REASONS]].to_numpy(dtype=int)
    np.testing.assert_allclose(predict_reason_models(classifier,X_valid),probs,rtol=1e-12,atol=1e-12)
    np.testing.assert_array_equal(label_predictions(probs)[0],prediction)
    np.testing.assert_allclose(f1_score(truth,prediction,average="samples",zero_division=0),row.sample_f1,atol=1e-14)
    np.testing.assert_allclose(f1_score(truth,prediction,average="macro",zero_division=0),row.macro_f1,atol=1e-14)
    packet=joblib.load(OUT/"labeling_models"/f"{row.experiment}.joblib")
    matrix=transform_clustering(packet,train)
    archive=np.load(OUT/"clustering_inputs"/f"{row.representation}.npz")
    assert archive['ID'].tolist()==train.ID.tolist()
    np.testing.assert_allclose(matrix,archive['coordinates'],rtol=1e-12,atol=1e-12)
    if row.family=="gmm":y,w,_,_,_=partial_labels(packet["clustering"].predict_proba(matrix))
    else:
        y,w,_,_,_,_=dbscan_labels(packet["clustering"])
        with threadpool_limits(limits=1):
            refit=clone(packet["clustering"]).fit(matrix)
        np.testing.assert_array_equal(refit.labels_,packet["clustering"].labels_)
    dataset=pd.read_csv(OUT/"datasets"/f"{row.experiment}.csv",float_precision="round_trip")
    assert dataset.ID.tolist()==train.ID.tolist()
    np.testing.assert_array_equal(y,dataset[REASONS].to_numpy(dtype=int))
    np.testing.assert_allclose(w,dataset[[f"weight_{r}" for r in REASONS]].to_numpy(),rtol=1e-12,atol=1e-14)
# Independently refit the selected classifier from its persisted 150-dimensional training dataset.
frame=pd.read_csv(OUT/"datasets"/f"{best.experiment}.csv",float_precision="round_trip")
scaled=classifier_scaler.transform(frame[FEATURES].to_numpy());replay={**selected,"estimators":[]}
with threadpool_limits(limits=1):
    for j,r in enumerate(REASONS):
        y=frame[r].to_numpy(dtype=int);w=frame[f"weight_{r}"].to_numpy();included=w>0
        estimator=clone(selected["estimators"][j])
        with warnings.catch_warnings():
            warnings.simplefilter("error",ConvergenceWarning)
            estimator.fit(scaled[included],y[included],sample_weight=w[included]*157/w[included].sum())
        replay["estimators"].append(estimator)
np.testing.assert_allclose(predict_reason_models(replay,X_valid),selected_probability,rtol=1e-8,atol=1e-10)
np.testing.assert_array_equal(label_predictions(predict_reason_models(replay,X_valid))[0],label_predictions(selected_probability)[0])
for rel,h in protected_hashes.items():assert hashlib.sha256((DATA/rel).read_bytes()).hexdigest()==h
print("All 28 saved labelers/classifiers, input replay, DBSCAN refits, reference controls, score checks, and independent selected-classifier refit passed.")
display(best_by_representation[["family","representation","relative_scale","sample_f1","macro_f1","generated_trials","delta_vs_current_leader"]].round(6))


All 28 saved labelers/classifiers, input replay, DBSCAN refits, reference controls, score checks, and independent selected-classifier refit passed.


,family,representation,relative_scale,sample_f1,macro_f1,generated_trials,delta_vs_current_leader
1,dbscan,embeddings_only,0.00,0.910256,0.811421,133,0.000000
23,dbscan,criteria_scale0p25,0.25,0.910256,0.815391,469,0.000000
14,dbscan,diseases_scale0p25,0.25,0.910256,0.815391,515,0.000000
0,gmm,embeddings_only,0.00,0.905983,0.811626,696,-0.004274
25,dbscan,criteria_scale1,1.00,0.901709,0.803884,268,-0.008547
10,dbscan,diseases_scale0p1,0.10,0.897436,0.803884,168,-0.012821
19,dbscan,criteria_scale0p1,0.10,0.893162,0.792111,171,-0.017094
4,gmm,diseases_scale0p1,0.10,0.893162,0.807626,871,-0.017094
16,dbscan,diseases_scale1,1.00,0.893162,0.804324,0,-0.017094
7,gmm,criteria_scale0p1,0.10,0.888889,0.802384,876,-0.021368


In [9]:
candidate=None
if best.sample_f1>db_metrics["sample_f1"]+1e-12:
    test=pd.read_csv(INPUT_DIR/"part1_test.csv");sample=pd.read_csv(INPUT_DIR/"part1_sample_submission.csv")
    assert test.ID.is_unique and sample.ID.is_unique and set(test.ID)==set(sample.ID)
    assert set(test.ID).isdisjoint(train.ID) and set(test.ID).isdisjoint(valid.ID)
    prediction,empty=label_predictions(predict_reason_models(selected,test[FEATURES].to_numpy()))
    output=pd.DataFrame({"ID":test.ID,"Predicted":[" ".join(r for r,yes in zip(REASONS,row) if yes) for row in prediction]})
    output=output.set_index("ID").loc[sample.ID].reset_index()
    assert output.shape==(312,2) and output.Predicted.str.len().gt(0).all()
    path=OUT/"submissions"/f"part1_{best.experiment}_logistic_C1000_submission.csv"
    output.to_csv(path,index=False);pd.testing.assert_frame_equal(pd.read_csv(path),output)
    candidate={"experiment":best.experiment,"validation_sample_f1":float(best.sample_f1),"validation_macro_f1":float(best.macro_f1),
               "submission_file":str(path.relative_to(OUT)),"submission_sha256":hashlib.sha256(path.read_bytes()).hexdigest(),
               "model_file":f"classifier_models/{best.experiment}.joblib","model_sha256":hashlib.sha256((OUT/"classifier_models"/f"{best.experiment}.joblib").read_bytes()).hexdigest(),
               "test_rows":312,"test_fallback_trials":int(empty.sum()),"kaggle_score":None,"kaggle_status":"prepared locally; not submitted"}
    (OUT/"submission_manifest.json").write_text(json.dumps(candidate,indent=2)+"\n")
settings={"context_scales":CONTEXT_SCALES,"context_groups":["diseases","criteria"],"representations":list(REPRESENTATIONS),
          "context_normalization":"training mean centering; aggregate RMS matched to embedding RMS before relative multiplier",
          "context_variance_relative_to_embeddings":[scale**2 for scale in CONTEXT_SCALES],
          "disease_source_representation":"diseases_min5","criteria_source_representation":"criteria_min3",
          "criteria_svd_components":TEXT_SVD_COMPONENTS,"criteria_svd_fit_rows":1171,"new_preprocessing_fit_rows":1171,
          "source_vocabularies_refitted":False,"gmm_parameters":gmm_packet["clustering"].get_params(),
          "dbscan_min_samples":MIN_SAMPLES,"dbscan_radius_quantiles":DBSCAN_QUANTILES,"dbscan_radius_training_only":True,
          "configurations":28,"gmm_configurations":7,"dbscan_configurations":21,
          "classifier_features":FEATURES,"extra_classifier_features":False,"classifier_C":1000,
          "classifier_scaler_fit_rows":157,"clustering_scaler_fit_rows":1171,"fitting_weight_total_per_reason":157,
          "label_rule":LABEL_RULE,"known_targets_preserved":True,"known_raw_weight":1,"cutoff":0.5,"fallback":True,
          "validation_rows":78,"validation_used_for_fit":False,"test_used_for_fit":False,
          "reference_gmm_sample_f1":gmm_reference_metrics["sample_f1"],"reference_dbscan_sample_f1":db_metrics["sample_f1"],
          "best_experiment":best.experiment,"best_sample_f1":float(best.sample_f1),"best_macro_f1":float(best.macro_f1),
          "leader_changed":bool(best.sample_f1>db_metrics["sample_f1"]+1e-12),"candidate":candidate,
          "protected_input_sha256":protected_hashes,
          "dataset_file_sha256":{str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted((OUT/"datasets").glob('*.csv'))},
          "selection":"sample F1; ties prefer current leader, fewer clustering dimensions, then lower added scale",
          "versions":{"python":platform.python_version(),"sklearn":sklearn.__version__,"numpy":np.__version__,"pandas":pd.__version__},"new_kaggle_scores":None}
(OUT/"settings.json").write_text(json.dumps(settings,indent=2)+"\n")
def result_table(rows):
    lines=["| Family | Representation | Radius quantile | Sample F1 | Macro F1 | Generated trials | Change vs leader |",
           "|---|---|---:|---:|---:|---:|---:|"]
    for row in rows.itertuples(index=False):
        quantile=f"{row.eps_quantile:g}" if row.family=="dbscan" else "—"
        lines.append(f"| {row.family} | {row.representation} | {quantile} | {row.sample_f1:.6f} | {row.macro_f1:.6f} | {row.generated_trials} | {row.delta_vs_current_leader:+.6f} |")
    return "\n".join(lines)
lines=["# Added features in GMM and DBSCAN clustering","",
       "Seven clustering representations cross embeddings-only with disease or criteria blocks at relative scales 0.10/0.25/1. "
       "GMM uses the original K=10 tied-covariance configuration (7 fits). DBSCAN uses min_samples=5 and epsilon at the "
       "25th/50th/75th percentiles of each representation's training fifth-neighbor distances, including self as in the original "
       "implementation (21 fits). Total 28 settings. All fit on the 1,171 training trials only.","",
       "## Input preprocessing and controls","",
       "Disease input reuses the 71-indicator diseases_min5 encoding. Criteria reuses the existing 2,000-term criteria_min3 "
       "TF-IDF vocabulary/IDF, with a new 20-component training-only TruncatedSVD (seed 42, n_iter=7) for manageable GMM "
       "covariances. Added blocks are centered using their training means and normalized to match the aggregate training "
       "RMS of the original standardized embedding block before applying the relative scale. Scales 0.10/0.25/1 add "
       "aggregate variance equal to 1%/6.25%/100% of the embedding variance. They control input geometry and covariance "
       "regularization, not literal GMM likelihood feature-importance weights.","",
       "Every final classifier still uses only the original 150 embeddings, the same frozen 157-row scaler, C=1000, "
       "normalized fitting-weight total 157 per reason, cutoff 0.5, and nonempty fallback. Generated labels/weights can "
       "change with clusters, but known labels and the label acceptance/weight-budget rules are fixed. Validation/test "
       "never enter preprocessing or clustering. At inference only the original embeddings are needed by the classifier.","",
       f"Reference sample F1: original GMM **{gmm_reference_metrics['sample_f1']:.6f}**, current DBSCAN **{db_metrics['sample_f1']:.6f}**. "
       f"Best setting: **{best.experiment}**, sample F1 **{best.sample_f1:.6f}**, macro F1 **{best.macro_f1:.6f}**.","",
       "## Best by representation","",result_table(best_by_representation),"",
       "## All configurations","",result_table(comparison),"",
       "## Artifacts and verification","",
       "Both original family controls reproduce their targets, weights, and validation probabilities. Every saved labeling "
       "packet recreates its exact clustering coordinates and targets/weights; DBSCAN is independently refitted with "
       "matching cluster assignments. All saved classifiers reload and rescore. The selected classifier independently "
       "refits from its saved dataset with matching probabilities; all classifier estimators have exactly 150 input features. "
       "Protected source data and models retain their checksums.","",
       "representations/, profiles/, and clustering_inputs/ preserve encoders, SVD, normalization, geometry, training IDs, "
       "coordinates, and neighbor distances. labeling_models/, clusters/, clustering_outputs/, datasets/, label_audits/, "
       "classifier_models/, and validation_predictions/ preserve all fits and generated labels/weights. comparison.csv "
       "includes matched family and matched radius-quantile controls; dbscan_radius_audit.csv documents each recomputed "
       "radius. best_by_family_group.csv, best_by_representation.csv, label_counts.csv, and settings.json record results and checksums.","",
       "This bounded screen evaluates these encodings, scales, and density settings on the repeatedly consulted 78-trial "
       "validation set; it does not exhaust all feature representations or establish independent test improvement. "
       "No self-training, final validation-label refit, Kaggle submission, commit, or tag was performed.",""]
lines.append(f"New candidate prepared locally: `{candidate['submission_file']}`. Kaggle score unknown." if candidate else
             "No setting beats the current leader; retain the original embeddings-only DBSCAN classifier.")
(OUT/"SUMMARY.md").write_text("\n".join(lines)+"\n")
print("Best:",best.experiment,"sample F1:",best.sample_f1,"Leader changed:",settings['leader_changed'])
print("Candidate:",candidate)


Best: dbscan_embeddings_only_q25 sample F1: 0.9102564102564102 Leader changed: False
Candidate: None
